# 📊 TP02 – How to Work with Big Data Files (5GB+) in Python

This notebook demonstrates different techniques to handle large CSV files (≥5GB) efficiently in **Python** using:
- `pandas` with chunksize
- `Dask` for parallel computation
- Compression techniques (`gzip`, `bz2`, etc.)

We’ll use the **Telecommunication Activity over Milano** dataset, which contains aggregated Call Detail Records (CDR) such as SMS, calls, and internet usage per grid/time interval.


In [1]:
# 🧩 Step 1: Import required libraries
import numpy as np
import pandas as pd
import dask.dataframe as dd
import os
import time
import psutil  
import shutil  
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/telecom/data2.csv
/kaggle/input/telecom/data1.csv


## 🧠 Step 2: Load the dataset using pandas (normal read)

We’ll first load the dataset directly using `pandas.read_csv()` to observe:
- Load time  
- Memory usage  


In [2]:
start_time = time.time()

file_path = '/kaggle/input/telecom/data1.csv'

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)

try:
    df = pd.read_csv(file_path)
    print(df.info())
except MemoryError:
    print("❌ MemoryError: File too large for direct read!")

mem_after = process.memory_info().rss / (1024 ** 2)
end_time = time.time()

print(f"⏱️ Load time: {end_time - start_time:.2f} seconds")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160108003 entries, 0 to 160108002
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   GridID        int64  
 1   TimeInterval  int64  
 2   countrycode   int64  
 3   smsin         float64
 4   smsout        float64
 5   callin        float64
 6   callout       float64
 7   internet      float64
dtypes: float64(5), int64(3)
memory usage: 9.5 GB
None
⏱️ Load time: 252.60 seconds
💾 Memory used: 9777.79 MB


## ⚙️ Step 3: Read CSV using pandas chunks (memory-efficient)

`chunksize` allows pandas to read the dataset in smaller portions, processing one chunk at a time without loading the entire file into memory.


##### Chunksize = 100_000 ---------------------------------------------------------------------------------------------------------------------------------

In [8]:
start_time = time.time()

chunksize = 100_000
chunk_list = []

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)


for chunk in pd.read_csv(file_path, chunksize=chunksize):
    chunk_mean = chunk['internet'].mean()
    chunk_list.append(chunk_mean)

mem_after = process.memory_info().rss / (1024 ** 2)


average_internet = np.mean(chunk_list)
end_time = time.time()

print(f"🌐 Average Internet Usage (calculated via chunks): {average_internet}")
print(f"⏱️ Load time (chunks): {end_time - start_time:.2f} seconds")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")


🌐 Average Internet Usage (calculated via chunks): 37.476895511631916
⏱️ Load time (chunks): 157.83 seconds
💾 Memory used: -16.16 MB


   ##### Chunksize = 50_000 ----------------------------------------------------------------------------------------------------------------------------------

In [6]:
start_time = time.time()

chunksize = 50_000
chunk_list = []

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)


for chunk in pd.read_csv(file_path, chunksize=chunksize):
    chunk_mean = chunk['internet'].mean()
    chunk_list.append(chunk_mean)

mem_after = process.memory_info().rss / (1024 ** 2)


average_internet = np.mean(chunk_list)
end_time = time.time()

print(f"🌐 Average Internet Usage (calculated via chunks): {average_internet}")
print(f"⏱️ Load time (chunks): {end_time - start_time:.2f} seconds")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")


🌐 Average Internet Usage (calculated via chunks): 37.72785744127729
⏱️ Load time (chunks): 164.47 seconds
💾 Memory used: 15.68 MB


##### Chunksize = 10_000 ----------------------------------------------------------------------------------------------------------------------------------

In [7]:
start_time = time.time()

chunksize = 10_000
chunk_list = []

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)


for chunk in pd.read_csv(file_path, chunksize=chunksize):
    chunk_mean = chunk['internet'].mean()
    chunk_list.append(chunk_mean)

mem_after = process.memory_info().rss / (1024 ** 2)


average_internet = np.mean(chunk_list)
end_time = time.time()

print(f"🌐 Average Internet Usage (calculated via chunks): {average_internet}")
print(f"⏱️ Load time (chunks): {end_time - start_time:.2f} seconds")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")


🌐 Average Internet Usage (calculated via chunks): 41.13927723093188
⏱️ Load time (chunks): 163.59 seconds
💾 Memory used: 1.37 MB


## ⚡ Step 4: Use Dask 

`Dask` can handle large datasets by splitting the file into partitions and processing them in parallel. It’s more scalable than pandas for large data.


In [9]:
start_time = time.time()

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)

dask_df = dd.read_csv(file_path)
print(dask_df.head())

internet_mean = dask_df['internet'].mean().compute()
mem_after = process.memory_info().rss / (1024 ** 2)

end_time = time.time()
print(f"🌐 Average Internet Usage (Dask): {internet_mean}")
print(f"⏱️ Load + compute time (Dask): {end_time - start_time:.2f} seconds")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


   GridID   TimeInterval  countrycode     smsin    smsout    callin   callout  \
0       1  1383260400000            0  0.081363       NaN       NaN       NaN   
1       1  1383260400000           39  0.141864  0.156787  0.160938  0.052275   
2       1  1383261000000            0  0.136588       NaN       NaN  0.027300   
3       1  1383261000000           33       NaN       NaN       NaN       NaN   
4       1  1383261000000           39  0.278452  0.119926  0.188777  0.133637   

    internet  
0        NaN  
1  11.028366  
2        NaN  
3   0.026137  
4  11.100963  
🌐 Average Internet Usage (Dask): 36.773472693448134
⏱️ Load + compute time (Dask): 74.99 seconds
💾 Memory used: 900.77 MB


## 🗜️ Step 5: Apply compression and compare performance

We’ll compress the dataset using `gzip`, then measure:
- File size before and after compression  
- Load time for compressed file  


In [10]:

start_time = time.time()

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)

sample_df = df

compressed_path = '/kaggle/working/telecom_dd_compressed.csv.gz'

sample_df.to_csv(compressed_path, index=False, compression='gzip')

mem_after = process.memory_info().rss / (1024 ** 2)

end_time = time.time()

print(f"✅ Sample compressed successfully in {end_time - start_time:.2f} seconds")

original_size = os.path.getsize(file_path) / (1024 ** 2)
compressed_size = os.path.getsize(compressed_path) / (1024 ** 2)

print(f"📁 Original file size: {original_size:.2f} MB")
print(f"📦 Compressed sample size: {compressed_size:.2f} MB")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")

✅ Sample compressed successfully in 2721.10 seconds
📁 Original file size: 9840.67 MB
📦 Compressed sample size: 2471.86 MB
💾 Memory used: 1.88 MB


In [11]:
end_time = time.time()
print(f"⏱️ Load time (compressed gzip): {end_time - start_time:.2f} seconds")

⏱️ Load time (compressed gzip): 3290.14 seconds


## 📈 Step 7: Compare all methods

| Method | Load Time (s) | Memory Usage (GB) |  Notes |
|---------|----------------|-------------------|----------------|
| pandas (normal) | 207.87 seconds | 9.5 GB | crash in the case when ram is less then 8GB |
| pandas (chunks) | 201.98 seconds | o.10 GB |  |
| Dask | 83.68 seconds | 0.84 GB |   |
| gzip compression | 1495.36 seconds | 1.27 GB | 📁 Original file size: 9840.67 MB 📦 Compressed sample size: 2471.86 MB |

---

## ✅ Conclusion

- `pandas.read_csv()` is simple but unsuitable for very large files especially on low resources devices.  
- `chunksize` helps process data sequentially without exceeding memory.  
- `Dask` is best for big datasets — it handles partitions and parallelism automatically.  
- Compression (`gzip`) saves disk space, though it slightly increases load time.




In [14]:
# 🗜️ Compression on dask files
import os
import glob 

start_time = time.time()

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 ** 2)


sample_df = dask_df

compressed_path = '/kaggle/working/telecom_compressed.csv.gz'

sample_df.to_csv(compressed_path, index=False, compression='gzip')

mem_after = process.memory_info().rss / (1024 ** 2)
end_time = time.time()

print(f"✅ Sample compressed successfully in {end_time - start_time:.2f} seconds")

original_size = os.path.getsize(file_path) / (1024 ** 2)
compressed_size = os.path.getsize(compressed_path) / (1024 ** 2)

compressed_files = glob.glob(f'{compressed_path}/*.part')


total_compressed_size_bytes = sum(os.path.getsize(f) for f in compressed_files)

total_compressed_size_mb = total_compressed_size_bytes / (1024 ** 2)


print(f"📁 Original file size: {original_size:.2f} MB")
print(f"📦 Total size of all compressed files: {total_compressed_size_mb:.2f} MB")
print(f"💾 Memory used: {mem_after - mem_before:.2f} MB")

✅ Sample compressed successfully in 1456.26 seconds
📁 Original file size: 9840.67 MB
📦 Total size of all compressed files: 2471.97 MB
💾 Memory used: 45.38 MB
